# Open, encode, decode, and visualize a triangle-mesh sequence

This uses the real `4d_files/Rafa_Approves_hd_4k` OBJ sequence and attempts every codec registered in the current environment. Optional source-only and native-process codecs report missing prerequisites instead of disappearing from the results. The current canonical value is a finite `TriangleMesh` sequence; point clouds, volumes, Gaussian splats, and live streams are not part of this API slice.

In [ ]:
import os
import time
from pathlib import Path
import numpy as np

from open4d import MemoryFrameProvider, Sequence
from open4d.codec import available_codecs, decode_sequence, encode_sequence
from open4d.io import inspect_sequence, open_sequence
from open4d.visualization import visualize

REPOSITORY = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
DATASET = Path(os.environ["OPEN4D_DATASET"]) if os.environ.get("OPEN4D_DATASET") else next(path / "4d_files/Rafa_Approves_hd_4k" for path in (Path.cwd(), *Path.cwd().parents) if (path / "4d_files/Rafa_Approves_hd_4k").is_dir())
ARTIFACT_DIR = Path(os.environ.get("OPEN4D_ARTIFACT_DIR", REPOSITORY / ".context/rafa_codecs"))
DEMO_FRAMES = int(os.environ.get("OPEN4D_DEMO_FRAMES", "2")) or None
DEVICE = os.environ.get("OPEN4D_NOTEBOOK_DEVICE", "auto")  # auto: CUDA, then Apple Metal/MPS, then CPU
REQUIRE_ALL = os.environ.get("OPEN4D_NOTEBOOK_REQUIRE_ALL") == "1"
CODEC_INFOS = {info.id: info for info in available_codecs()}
CODECS = tuple(CODEC_INFOS)
ENCODE_OPTIONS = {
    "klt": dict(resolution=63, num_components=32, block_size=8, k_total=4096, training_frames=(0,)),
    "n4mc": dict(resolution=127, epochs=100, hidden_channels=(8, 16), latent_channels=8, learning_rate=3e-3, device=DEVICE),
    "qndf": dict(coarse_size=10000, num_subdiv=0, epochs=20, hidden_dim=8, num_layers=3, batch_size=1024, device=DEVICE),
    "qndf-int8": dict(coarse_size=10000, num_subdiv=0, epochs=20, hidden_dim=8, num_layers=3, batch_size=1024, device=DEVICE),
    "temporal-delta": dict(face_budget=10000, quantization_bits=12),
    "temporal-pca": dict(face_budget=10000, quantization_bits=12, components=3),
}
QUALITY_RMS_LIMIT = 0.08
QUALITY_COMPONENT_LIMIT = 4
QUALITY_MIN_TRIANGLES = 3000
DECODE_OPTIONS = {"n4mc": {"device": DEVICE, "min_component_faces": 32}, "qndf": {"device": DEVICE}}
DECODE_OPTIONS["klt"] = {"device": "cpu"}
DECODE_OPTIONS["qndf-int8"] = {"device": "cpu"}

def options_for(codec):
    encode = dict(ENCODE_OPTIONS.get(codec, {}))
    decode = dict(DECODE_OPTIONS.get(codec, {}))
    if codec in {"vdmc", "faster_vdmc"}:
        prefix = f"OPEN4D_{codec.upper()}"
        names = (f"{prefix}_ENCODER", f"{prefix}_DECODER")
        app_dirs = [REPOSITORY / "open4d/codecs" / codec / "build/Release" / leaf for leaf in ("source/app", "bin")]
        encoder = os.environ.get(names[0]) or next((path / "encode" for path in app_dirs if (path / "encode").is_file()), None)
        decoder = os.environ.get(names[1]) or next((path / "decode" for path in app_dirs if (path / "decode").is_file()), None)
        if not encoder or not decoder:
            raise RuntimeError(f"build {codec} or set {', '.join(names)}")
        encode.update(encoder=encoder)
        decode.update(decoder=decoder)
    return encode, decode

def geometry_quality(expected, actual, seed):
    if np.array_equal(expected.positions, actual.positions) and np.array_equal(expected.triangles, actual.triangles):
        expected_cloud, distances = expected.positions, np.zeros(1)
    else:
        try:
            import point_cloud_utils as pcu
            from scipy.spatial import cKDTree
            clouds = []
            for mesh in (expected, actual):
                vertices, faces = np.asarray(mesh.positions, dtype=np.float64), np.asarray(mesh.triangles, dtype=np.int32)
                indices, barycentric = pcu.sample_mesh_random(vertices, faces, 2000, random_seed=seed)
                clouds.append(np.einsum("nij,ni->nj", vertices[faces[indices]], barycentric))
            expected_cloud, actual_cloud = clouds
            distances = np.concatenate((cKDTree(actual_cloud).query(expected_cloud)[0], cKDTree(expected_cloud).query(actual_cloud)[0]))
        except ImportError:
            clouds = [np.asarray(mesh.positions)[np.linspace(0, len(mesh.positions) - 1, min(1000, len(mesh.positions)), dtype=int)] for mesh in (expected, actual)]
            expected_cloud, actual_cloud = clouds
            nearest = lambda left, right: np.sqrt(np.min(np.sum((left[:, None] - right[None]) ** 2, axis=2), axis=1))
            distances = np.concatenate((nearest(expected_cloud, actual_cloud), nearest(actual_cloud, expected_cloud)))
    rms = np.sqrt(np.mean(distances ** 2)) / np.linalg.norm(np.ptp(expected_cloud, axis=0))
    parent = np.arange(len(actual.positions))
    def find(value):
        while parent[value] != value: parent[value] = parent[parent[value]]; value = parent[value]
        return value
    for first, second, third in actual.triangles:
        for left, right in ((first, second), (first, third)):
            left, right = find(left), find(right)
            if left != right: parent[right] = left
    roots = np.array([find(int(triangle[0])) for triangle in actual.triangles])
    component_faces = np.unique(roots, return_counts=True)[1]
    return float(rms), int(np.count_nonzero(component_faces >= max(32, len(actual.triangles) // 1000)))

In [ ]:
info = inspect_sequence(DATASET)
sequence = open_sequence(DATASET, fps=30)
selected = sequence if DEMO_FRAMES is None else sequence[:DEMO_FRAMES]
demo = Sequence(MemoryFrameProvider(
    tuple(selected), metadata=selected.metadata, topology=selected.topology,
    has_constant_vertex_count=selected.has_constant_vertex_count,
    has_vertex_correspondence=selected.has_vertex_correspondence,
))
print(f"Loaded {info.frame_count} {info.format.upper()} frames; using {len(demo)} for this run.")

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
results = []
decoded_sequences = {}
for codec in CODECS:
    codec_info = CODEC_INFOS[codec]
    artifact = ARTIFACT_DIR / f"rafa-{codec}{codec_info.suffixes[0]}"
    try:
        encode_options, decode_options = options_for(codec)
        start = time.perf_counter()
        encode_sequence(demo, artifact, codec=codec, overwrite=True, **encode_options)
        encode_s = time.perf_counter() - start
        start = time.perf_counter()
        candidate = decode_sequence(artifact, **decode_options)
        exact = True
        surface_errors, component_counts, triangle_counts = [], [], []
        assert candidate.metadata == demo.metadata and len(candidate) == len(demo)
        for ordinal, (expected, actual) in enumerate(zip(demo, candidate, strict=True)):
            assert (actual.frame_index, actual.timestamp, actual.metadata) == (expected.frame_index, expected.timestamp, expected.metadata)
            exact &= np.array_equal(actual.geometry.positions, expected.geometry.positions)
            exact &= np.array_equal(actual.geometry.triangles, expected.geometry.triangles)
            assert len(actual.geometry.positions) and len(actual.geometry.triangles)
            quality = geometry_quality(expected.geometry, actual.geometry, 100 + ordinal)
            surface_errors.append(quality[0]); component_counts.append(quality[1]); triangle_counts.append(len(actual.geometry.triangles))
        if codec_info.lossless:
            assert exact
        assert max(surface_errors) < QUALITY_RMS_LIMIT
        assert max(component_counts) <= QUALITY_COMPONENT_LIMIT
        assert min(triangle_counts) >= QUALITY_MIN_TRIANGLES
        decode_s = time.perf_counter() - start
    except Exception as error:
        results.append(dict(codec=codec, status="error", detail=f"{type(error).__name__}: {error}"))
    else:
        results.append(dict(codec=codec, status="ok", size=artifact.stat().st_size, encode_s=encode_s, decode_s=decode_s, exact=exact, surface_rms=max(surface_errors), components=max(component_counts), triangles=min(triangle_counts)))
        decoded_sequences[codec] = candidate

In [ ]:
print("codec                 status       size (MB)  encode (s)  decode+verify (s)  surface RMS  components  triangles")
for row in results:
    if row["status"] == "ok":
        print(f"{row['codec']:<21} ok           {row['size'] / 1_000_000:>9.2f}  {row['encode_s']:>10.3f}  {row['decode_s']:>17.3f}  {row['surface_rms']:>11.4f}  {row['components']:>10}  {row['triangles']:>9}")
    else:
        print(f"{row['codec']:<21} error        {row['detail']}")
print(f"Attempted all {len(CODECS)} registered codecs; {len(decoded_sequences)} decoded successfully.")
failed = [row['codec'] for row in results if row['status'] != 'ok']
if REQUIRE_ALL and failed:
    raise RuntimeError(f"registered codecs failed: {', '.join(failed)}")

In [ ]:
try:
    if os.environ.get("OPEN4D_NOTEBOOK_HEADLESS") == "1":
        print("Headless run: visualization calls skipped.")
    else:
        for codec, decoded in decoded_sequences.items():
            print(f"Visualizing {codec}; close its viewer to continue.")
            visualize(decoded, title=f"Open4D: {codec}", up="y", fps=30)
finally:
    for decoded in decoded_sequences.values():
        decoded.close()
    sequence.close()